## Importing the necessary dependencies. 

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline ,Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score



## Loading the data. 

In [ ]:
pd.set_option('display.max_columns', 100)
path = "../Data/AmesHousing.csv"
df = pd.read_csv(path)
df = df.drop(columns=['Order', 'PID', 'Fence', 'Misc Feature', 'Alley'])

## Categorical Columns - Assumption

In [ ]:
# Columns with < 10 unique values added to a dict(column: Uniques pair) - Regarded as Categorical. 
cat_cols = dict()
for column in [col for col in df.columns if df[col].nunique() <= 10]:
    unq = df[column].unique()
    cat_cols[column] = list(unq)


## Columns to be preprocessed - Imputed, Scaled, Encoded

In [ ]:
# Columns Using Ex, Gd quality Scale. - Ordinal Columns. 
qual_cols = []
for col in df.columns:
    unique_vals = df[col].dropna().unique()
    if set(unique_vals).issubset(set(["Ex", "Gd", "TA", "Fa", "Po"])):
        qual_cols.append(col)

# Nominal Columns - Not in qualitative(Ordinal) Columns and not a number!
nom_cols = [col for col in cat_cols.keys() if col not in qual_cols and col not in df.select_dtypes(include=['number']).columns.tolist()]
nom_cols += ['Neighborhood', 'Exterior 1st', 'Exterior 2nd']

#------------------------------------------------------------------------------------------------------
# --------------------Only Time The dataframe is directly affected during EDA. ------------------------
# Remove years Built and remod - create age and was_remodeled columns. 
df['Was_remodeled'] = (df['Year Remod/Add'] != df['Year Built']).astype(int)
df['age'] = 2026 - df['Year Built']
df = df.drop(columns=['Year Built', 'Year Remod/Add'])
#------------------------------------------------------------------------------------------------------

# Finding Columns to Scale. 
scale_cols = []
for col in df.columns:
    if col not in qual_cols + nom_cols and col not in cat_cols.keys():
        scale_cols.append(col)
scale_cols.remove('SalePrice')
scale_cols.remove('Was_remodeled')

# Finding Columns to Impute. 
null_cols = []
for col in df.columns:
    if df[col].isnull().sum() > 0:
        null_cols.append(col)
zero_imp_cols = ['Lot Frontage', 'Mas Vnr Area', 'BsmtFin SF 1', 'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF', 'Bsmt Full Bath', 'Bsmt Half Bath', 'Garage Area'] # Removed 'Garage Qual', 'Pool QC'
mode_imp_cols = ['Garage Yr Blt', 'Garage Cars'] # Added Garage Cars. 

## Making Pipelines and estimator objects. 

In [ ]:

ord_enc = OrdinalEncoder( # Fix #2. 
    categories=[["Ex", "Gd", "TA", "Fa", "Po"]]*len(qual_cols),
    handle_unknown='use_encoded_value', unknown_value=-1, 
    encoded_missing_value=-1
)
ohe_enc = OneHotEncoder(handle_unknown='ignore')
scaler = StandardScaler()

#------------------------BUG FIXE #3----------------------------------------------------------------------------------------------
zero_imp_scale = Pipeline([
    ('impute', SimpleImputer(strategy='constant', fill_value=0)),
    ('scale', StandardScaler())
])
mode_imp_scale = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('scale', StandardScaler())
])
scale_cols_clean = [col for col in scale_cols if col not in qual_cols and col not in zero_imp_cols and col not in mode_imp_cols]
#---------------------------------------------------------------------------------------------------------------------------------

X = df.drop('SalePrice', axis=1)
y = np.log1p(df['SalePrice'])


## Final Pipeline and Score checking. 

In [39]:
preprocessor = ColumnTransformer([ # Does all the steps in parallel, use pipeline for reuse(BUG FIX #3)
    ('imp_zero_scale', zero_imp_scale, zero_imp_cols),
    ('imp_mode_scale', mode_imp_scale, mode_imp_cols),
    ('ord_enc', ord_enc, qual_cols),
    ('ohe_enc', ohe_enc, nom_cols),
    ('scaler', scaler, scale_cols_clean)
],
    remainder='passthrough'
)

pipe = Pipeline([ # Fix #1, Pipeline instead of make_pipeline. 
    ('prep', preprocessor),
    ('lr', LinearRegression())
])

r2_scores = cross_val_score(pipe, X, y, cv=5, scoring='r2')
rmse_scores = -cross_val_score(pipe, X, y, cv=5, scoring='neg_root_mean_squared_error')

print("R2 per fold:      ", np.round(r2_scores, 4))
print("Mean R2:          ", round(r2_scores.mean(), 4))
print("Mean RMSE (log$): ", round(rmse_scores.mean(), 4))

R2 per fold:       [0.8942 0.8911 0.8471 0.8096 0.9058]
Mean R2:           0.8696
Mean RMSE (log$):  0.145


In [ ]:
"""
# -------------REMOVE DOCSSTRING COMMENT TO SAVE MODEL TO .JOBLIB FILE-----------------------
import joblib

# Fit the pipeline on the full dataset (X, y)
pipe.fit(X, y)

# Save the model to a joblib file
model_filename = 'ames_housing_model.joblib'
joblib.dump(pipe, model_filename)
print(f"Model saved successfully to {model_filename}")
"""

## Visualizations - Skip this. 

In [ ]:
"""
# --------------REMOVE DOCSTRING COMMENT AND RUN TO VIEW VISUALIZATIONS -------------------
# Missingness Map
sns.set_theme(style='whitegrid')
plt.figure(figsize=(10, 6))

# cmap='viridis' is colorblind-friendly. cbar=False removes the legend.
sns.heatmap(ames.isnull(), cbar=False, cmap='viridis', yticklabels=False)

plt.title('Missing Data Map (Yellow = Missing)', fontsize=14)
plt.show()
# Missingness Map(Filtered)
plt.figure(figsize=(10, 6))

# 1. Filter to keep ONLY columns that have at least one NaN
missing_cols = ames.columns[ames.isnull().any()]
df_missing_only = ames[missing_cols]

# 2. Plot the heatmap using the filtered DataFrame
sns.heatmap(
    df_missing_only.isnull(), 
    cbar=False, 
    cmap='viridis', 
    yticklabels=False
)

plt.title('Missing Data Map (Filtered)', fontsize=14)
plt.show()
# Skewness
target = 'SalePrice'
skewness = ames[target].skew()

plt.figure(figsize=(10, 6))
sns.histplot(ames[target], kde=True, color='blue', bins=40)

# Display skewness directly in the title
plt.title(f'Distribution of {target}\nSkewness: {skewness:.3f}', fontsize=14)
plt.xlabel(f'{target} ($)')
plt.ylabel('Frequency')
plt.show()
# Correlation heatmap
# Number of variables for heatmap
k = 5 

# 1. Filter out categorical columns to avoid warnings/errors
numeric_df = ames.select_dtypes(include=[np.number])

# 2. Find the top 'k' correlated features with the target
corrmat = numeric_df.corr()
top_cols = corrmat.nlargest(k, target)[target].index

# 3. Calculate correlation matrix for just those top features
cm = np.corrcoef(numeric_df[top_cols].values.T)

# 4. Plot the heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(
    cm, 
    annot=True,              # Show the actual correlation numbers
    fmt='.2f',               # Format to 2 decimal places
    cmap='coolwarm',         # Blue (negative) to Red (positive)
    yticklabels=top_cols, 
    xticklabels=top_cols,
    square=True,             # Keep the cells square
    linewidths=0.5
)

plt.title(f'Top {k} Predictors Correlation Matrix', fontsize=14)
plt.xticks(rotation=45, ha='right') # Rotate x labels for better readability
plt.show()
### Visualizing with matplotlib only. 
# Missingness Map
plt.figure(figsize=(10, 6))

# Use imshow to plot the boolean mask
plt.imshow(ames.isnull(), aspect='auto', cmap='viridis', interpolation='none')

# Clean up the axes to match the seaborn look
plt.yticks([]) # Remove y-axis ticks (row numbers)
plt.xticks(ticks=np.arange(len(ames.columns)), labels=ames.columns, rotation=45, ha='right')

plt.title('Missing Data Map (Yellow = Missing)', fontsize=14)
plt.tight_layout()
plt.show()
# Missingness map(Filtered)
plt.figure(figsize=(10, 6))

# 1. Filter to keep ONLY columns that have at least one NaN
missing_cols = ames.columns[ames.isnull().any()]
df_missing_only = ames[missing_cols]

# 2. Plot the boolean mask of the filtered DataFrame
plt.imshow(df_missing_only.isnull(), aspect='auto', cmap='viridis', interpolation='none')

# 3. Clean up the axes
plt.yticks([]) # Remove y-axis ticks (row numbers)

# Set x-ticks to match our newly filtered column list
plt.xticks(ticks=np.arange(len(missing_cols)), labels=missing_cols, rotation=45, ha='right')

plt.title('Missing Data Map (Filtered)', fontsize=14)
plt.tight_layout()
plt.show()
# Skeweness
target = 'SalePrice'
skewness = ames[target].skew()

plt.figure(figsize=(10, 6))

# 1. Plot the histogram (density=True normalizes it so the KDE curve scales correctly)
plt.hist(ames[target], bins=40, density=True, color='cornflowerblue', alpha=0.7, edgecolor='white')

# 2. Calculate and plot the KDE curve
kde = gaussian_kde(ames[target])
x_vals = np.linspace(ames[target].min(), ames[target].max(), 1000)
plt.plot(x_vals, kde(x_vals), color='darkblue', linewidth=2)

plt.title(f'Distribution of {target}\nSkewness: {skewness:.3f}', fontsize=14)
plt.xlabel(f'{target} ($)')
plt.ylabel('Density')
plt.show()
# Correlation Map
k = 5 
numeric_df = ames.select_dtypes(include=[np.number])
corrmat = numeric_df.corr()
top_cols = corrmat.nlargest(k, target)[target].index
cm = np.corrcoef(numeric_df[top_cols].values.T)

fig, ax = plt.subplots(figsize=(8, 6))

# 1. Plot the color grid
cax = ax.imshow(cm, cmap='coolwarm', vmin=-1, vmax=1)

# 2. Add the colorbar on the side
fig.colorbar(cax)

# 3. Set up the X and Y axes labels
ax.set_xticks(np.arange(len(top_cols)))
ax.set_yticks(np.arange(len(top_cols)))
ax.set_xticklabels(top_cols, rotation=45, ha='right')
ax.set_yticklabels(top_cols)

# 4. Loop over data dimensions and create text annotations
for i in range(len(top_cols)):
    for j in range(len(top_cols)):
        # Change text color to white if the square is too dark for black text
        text_color = "white" if abs(cm[i, j]) > 0.6 else "black"
        ax.text(j, i, f"{cm[i, j]:.2f}", 
                ha="center", va="center", color=text_color)

plt.title(f'Top {k} Predictors Correlation Matrix', fontsize=14)
plt.tight_layout()
plt.show()
"""